# 1. 전처리 과정

In [1]:
# ── Section 1. 라이브러리 임포트 ──────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pickle
import os
import shutil

print("라이브러리 임포트 완료")


라이브러리 임포트 완료


In [2]:
# ── Section 2. 데이터 로드 ────────────────────────────────────
# 업로드 파일 4개:
# Monday    → 학습용 (정상만)
# Tuesday   → 테스트 (Brute Force)
# Wednesday → 테스트 (DoS)
# Friday-DDoS → 테스트 (DDoS)

monday_df    = pd.read_csv('../data/CICIDS2017/Monday-WorkingHours.pcap_ISCX.csv')
tuesday_df   = pd.read_csv('../data/CICIDS2017/Tuesday-WorkingHours.pcap_ISCX.csv')
wednesday_df = pd.read_csv('../data/CICIDS2017/Wednesday-workingHours.pcap_ISCX.csv')
friday_df    = pd.read_csv('../data/CICIDS2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')

print("Monday shape   :", monday_df.shape)
print("Tuesday shape  :", tuesday_df.shape)
print("Wednesday shape:", wednesday_df.shape)
print("Friday shape   :", friday_df.shape)

Monday shape   : (529918, 79)
Tuesday shape  : (445909, 79)
Wednesday shape: (692703, 79)
Friday shape   : (225745, 79)


In [3]:
# ── Section 3. 컬럼명 공백 제거 ───────────────────────────────
for df in [monday_df, tuesday_df, wednesday_df, friday_df]:
    df.columns = df.columns.str.strip()

print("\nTuesday Label:\n",   tuesday_df['Label'].value_counts())
print("\nWednesday Label:\n", wednesday_df['Label'].value_counts())
print("\nFriday Label:\n",    friday_df['Label'].value_counts())


Tuesday Label:
 Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

Wednesday Label:
 Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64

Friday Label:
 Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


In [4]:
# ── Section 4. 클렌징 ─────────────────────────────────────────
drop_cols = ['Label', 'Flow ID', 'Source IP', 'Destination IP', 'Timestamp']

def clean(df, is_test=False):
    labels = None
    if is_test:
        labels = df['Label'].copy()
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    if is_test:
        labels = labels.loc[df.index].values
    return df, labels

monday_clean, _               = clean(monday_df,    is_test=False)
tuesday_clean, tuesday_labels = clean(tuesday_df,   is_test=True)
wednesday_clean, wed_labels   = clean(wednesday_df, is_test=True)
friday_clean, fri_labels      = clean(friday_df,    is_test=True)

print("Monday 클렌징 후  :", monday_clean.shape)
print("Tuesday 클렌징 후 :", tuesday_clean.shape)
print("Wednesday 클렌징 후:", wednesday_clean.shape)
print("Friday 클렌징 후  :", friday_clean.shape)


Monday 클렌징 후  : (529481, 78)
Tuesday 클렌징 후 : (445645, 78)
Wednesday 클렌징 후: (691406, 78)
Friday 클렌징 후  : (225711, 78)


In [5]:
# ── Section 5. 스케일링 ───────────────────────────────────────
scaler = MinMaxScaler()
scaler.fit(monday_clean)  # 정상 데이터로만 fit

monday_scaled    = scaler.transform(monday_clean)
tuesday_scaled   = scaler.transform(tuesday_clean)
wednesday_scaled = scaler.transform(wednesday_clean)
friday_scaled    = scaler.transform(friday_clean)

print("스케일링 완료")


스케일링 완료


In [ ]:
# ── Section 6. Sliding Window ─────────────────────────────────
def create_sequences(data, window_size):
    return np.array([data[i:i+window_size] for i in range(len(data) - window_size)])

W1 = 5   # 1차 방어선용
W2 = 20  # 2차 방어선용

# Window=5
X_tr_w5  = create_sequences(monday_scaled,    W1)
X_te_tue_w5 = create_sequences(tuesday_scaled,   W1)
X_te_wed_w5 = create_sequences(wednesday_scaled, W1)
X_te_fri_w5 = create_sequences(friday_scaled,    W1)
y_te_tue_w5 = tuesday_labels[W1:]
y_te_wed_w5 = wed_labels[W1:]
y_te_fri_w5 = fri_labels[W1:]

# Window=20
X_tr_w20 = create_sequences(monday_scaled,    W2)
X_te_tue_w20 = create_sequences(tuesday_scaled,   W2)
X_te_wed_w20 = create_sequences(wednesday_scaled, W2)
X_te_fri_w20 = create_sequences(friday_scaled,    W2)
y_te_tue_w20 = tuesday_labels[W2:]
y_te_wed_w20 = wed_labels[W2:]
y_te_fri_w20 = fri_labels[W2:]

print(f"\nW5  Train: {X_tr_w5.shape}")
print(f"W20 Train: {X_tr_w20.shape}")


In [ ]:
# ── Section 7. Train/Validation 분리 ─────────────────────────
X_train_w5,  X_val_w5  = train_test_split(X_tr_w5,  test_size=0.1, random_state=42)
X_train_w20, X_val_w20 = train_test_split(X_tr_w20, test_size=0.1, random_state=42)

print(f"\nW5  Train: {X_train_w5.shape}  Val: {X_val_w5.shape}")
print(f"W20 Train: {X_train_w20.shape}  Val: {X_val_w20.shape}")


In [ ]:
# ── Section 8. 저장 (모델 + 이미지만) ────────────────────────
# 전처리 데이터는 메모리에 유지 (저장 불필요)
# 모델과 그래프만 저장

os.makedirs('models', exist_ok=True)

# 스케일러만 저장 (용량 작음)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("스케일러 저장 완료")
print("전처리 데이터는 메모리 유지 → 바로 Section 02로 진행")

In [ ]:
# ── Section 9. 다운로드 ───────────────────────────────────────
shutil.make_archive('preprocessed', 'zip', 'preprocessed')
files.download('preprocessed.zip')
print("다운로드 시작!")

# 2. 1차 방어선 벤치마크 (MLP vs 1D-CNN vs GRU)

In [ ]:
# ============================================================
# 02_tier1_benchmark.ipynb
# DeepGuard - 1차 방어선 벤치마크 (MLP vs 1D-CNN vs GRU)
# Window Size = 5
# ============================================================

# ── Section 1. 라이브러리 임포트 ──────────────────────────────
import numpy as np
import pickle
import time
import os
from google.colab import files
from sklearn.metrics import classification_report, roc_auc_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, Flatten,
                                     Conv1D, MaxPooling1D,
                                     GRU, TimeDistributed, RepeatVector)
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

print("라이브러리 임포트 완료")

In [ ]:
# ── Section 2. 전처리 파일 로드 ───────────────────────────────
uploaded = files.upload()  # preprocessed.zip 업로드

import shutil
shutil.unpack_archive('preprocessed.zip', 'preprocessed')

X_train = np.load('preprocessed/X_train_w5.npy')
X_val   = np.load('preprocessed/X_val_w5.npy')

X_te_tue = np.load('preprocessed/X_te_tue_w5.npy')
X_te_wed = np.load('preprocessed/X_te_wed_w5.npy')
X_te_fri = np.load('preprocessed/X_te_fri_w5.npy')
y_te_tue = np.load('preprocessed/y_te_tue_w5.npy', allow_pickle=True)
y_te_wed = np.load('preprocessed/y_te_wed_w5.npy', allow_pickle=True)
y_te_fri = np.load('preprocessed/y_te_fri_w5.npy', allow_pickle=True)

TIMESTEPS = X_train.shape[1]  # 5
FEATURES  = X_train.shape[2]  # 78

print(f"Train: {X_train.shape} / Val: {X_val.shape}")
print(f"TIMESTEPS={TIMESTEPS}, FEATURES={FEATURES}")


In [ ]:
# ── Section 3. 임계값 산출 함수 ───────────────────────────────
def get_threshold(model, X_val, percentile=95):
    X_pred = model.predict(X_val, verbose=0)
    mse = np.mean(np.square(X_val - X_pred), axis=(1, 2))
    return np.percentile(mse, percentile)

def get_mse(model, X):
    X_pred = model.predict(X, verbose=0)
    return np.mean(np.square(X - X_pred), axis=(1, 2))


In [ ]:
# ── Section 4. 평가 함수 ──────────────────────────────────────
def evaluate(model, threshold, name):
    results = {}
    for X_te, y_te, attack in [
        (X_te_tue, y_te_tue, 'BruteForce'),
        (X_te_wed, y_te_wed, 'DoS'),
        (X_te_fri, y_te_fri, 'DDoS'),
    ]:
        mse = get_mse(model, X_te)
        y_pred = (mse > threshold).astype(int)
        y_true = (y_te != 'BENIGN').astype(int)

        report = classification_report(y_true, y_pred, output_dict=True)
        recall    = report['1']['recall']    if '1' in report else 0
        precision = report['1']['precision'] if '1' in report else 0
        f1        = report['1']['f1-score']  if '1' in report else 0

        results[attack] = {
            'recall': recall,
            'precision': precision,
            'f1': f1
        }
        print(f"  [{name}] {attack} → Recall: {recall:.3f}  Precision: {precision:.3f}  F1: {f1:.3f}")
    return results

In [ ]:
# ── Section 5. EarlyStopping 공통 설정 ───────────────────────
es = EarlyStopping(monitor='val_loss', patience=5,
                   restore_best_weights=True)

In [ ]:
# ── Section 6. MLP Autoencoder ───────────────────────────────
print("\n========== EXP1: MLP ==========")

inp = Input(shape=(TIMESTEPS, FEATURES))
x   = Flatten()(inp)
x   = Dense(128, activation='relu')(x)
x   = Dense(64,  activation='relu')(x)
x   = Dense(128, activation='relu')(x)
out = Dense(TIMESTEPS * FEATURES, activation='sigmoid')(x)
from tensorflow.keras.layers import Reshape
out = Reshape((TIMESTEPS, FEATURES))(out)
mlp_model = Model(inp, out)
mlp_model.compile(optimizer='adam', loss='mse')
mlp_model.summary()

start = time.time()
mlp_hist = mlp_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
mlp_time = (time.time() - start) / len(mlp_hist.history['loss'])
mlp_threshold = get_threshold(mlp_model, X_val)
print(f"MLP 임계값: {mlp_threshold:.6f} / 에폭당 평균: {mlp_time:.2f}s")
mlp_results = evaluate(mlp_model, mlp_threshold, 'MLP')

In [ ]:
# ── Section 7. 1D-CNN Autoencoder ────────────────────────────
print("\n========== EXP2: 1D-CNN ==========")

inp = Input(shape=(TIMESTEPS, FEATURES))
x   = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inp)
x   = Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
x   = Conv1D(64, kernel_size=3, activation='relu', padding='same')(x)
out = TimeDistributed(Dense(FEATURES))(x)
cnn_model = Model(inp, out)
cnn_model.compile(optimizer='adam', loss='mse')
cnn_model.summary()

start = time.time()
cnn_hist = cnn_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
cnn_time = (time.time() - start) / len(cnn_hist.history['loss'])
cnn_threshold = get_threshold(cnn_model, X_val)
print(f"CNN 임계값: {cnn_threshold:.6f} / 에폭당 평균: {cnn_time:.2f}s")
cnn_results = evaluate(cnn_model, cnn_threshold, '1D-CNN')

In [ ]:
# ── Section 8. GRU Autoencoder ───────────────────────────────
print("\n========== EXP3: GRU ==========")

inp     = Input(shape=(TIMESTEPS, FEATURES))
encoded = GRU(64, activation='relu', return_sequences=False)(inp)
repeated= RepeatVector(TIMESTEPS)(encoded)
decoded = GRU(64, activation='relu', return_sequences=True)(repeated)
out     = TimeDistributed(Dense(FEATURES))(decoded)
gru_model = Model(inp, out)
gru_model.compile(optimizer='adam', loss='mse')
gru_model.summary()

start = time.time()
gru_hist = gru_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
gru_time = (time.time() - start) / len(gru_hist.history['loss'])
gru_threshold = get_threshold(gru_model, X_val)
print(f"GRU 임계값: {gru_threshold:.6f} / 에폭당 평균: {gru_time:.2f}s")
gru_results = evaluate(gru_model, gru_threshold, 'GRU')

In [ ]:
# ── Section 9. Graph 1. Loss Curve ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, hist, name in zip(axes,
    [mlp_hist, cnn_hist, gru_hist],
    ['MLP', '1D-CNN', 'GRU']):
    ax.plot(hist.history['loss'],     label='Train Loss')
    ax.plot(hist.history['val_loss'], label='Val Loss')
    ax.set_title(f'{name} Loss Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend()
plt.tight_layout()
plt.savefig('loss_curve_tier1.png', dpi=150)
plt.show()

In [ ]:
# ── Section 10. Graph 3. Scatter (Speed vs Recall) ────────────
avg_recall = lambda r: np.mean([r[k]['recall'] for k in r])

names  = ['MLP', '1D-CNN', 'GRU']
times  = [mlp_time, cnn_time, gru_time]
recalls= [avg_recall(mlp_results),
          avg_recall(cnn_results),
          avg_recall(gru_results)]
colors = ['steelblue', 'tomato', 'seagreen']

plt.figure(figsize=(7, 5))
for n, t, r, c in zip(names, times, recalls, colors):
    plt.scatter(t, r, s=200, color=c, label=n, zorder=5)
    plt.annotate(n, (t, r), textcoords='offset points',
                 xytext=(8, 4), fontsize=12)
plt.xlabel('에폭당 추론 시간 (s)')
plt.ylabel('평균 Recall')
plt.title('1차 방어선: Speed vs Recall')
plt.legend()
plt.tight_layout()
plt.savefig('scatter_tier1.png', dpi=150)
plt.show()

In [ ]:
# ── Section 11. 모델 저장 + 다운로드 ─────────────────────────
os.makedirs('models', exist_ok=True)
mlp_model.save('models/mlp_model.keras')
cnn_model.save('models/cnn_model.keras')
gru_model.save('models/gru_model.keras')

np.save('models/mlp_threshold.npy', mlp_threshold)
np.save('models/cnn_threshold.npy', cnn_threshold)
np.save('models/gru_threshold.npy', gru_threshold)

shutil.make_archive('tier1_models', 'zip', 'models')
files.download('tier1_models.zip')
files.download('loss_curve_tier1.png')
files.download('scatter_tier1.png')
print("저장 및 다운로드 완료!")

#  2차 방어선 벤치마크 (LSTM vs Bi-LSTM)
Window Size = 5


In [ ]:
# ── Section 1. 라이브러리 임포트 ──────────────────────────────
import numpy as np
import pickle
import time
import os
import shutil
from google.colab import files
from sklearn.metrics import classification_report
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM,
                                     Bidirectional, TimeDistributed,
                                     RepeatVector)
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

print("라이브러리 임포트 완료")

In [ ]:
# ── Section 2. 전처리 파일 로드 ───────────────────────────────
uploaded = files.upload()  # preprocessed.zip 업로드

shutil.unpack_archive('preprocessed.zip', 'preprocessed')

X_train = np.load('preprocessed/X_train_w20.npy')
X_val   = np.load('preprocessed/X_val_w20.npy')

X_te_tue = np.load('preprocessed/X_te_tue_w20.npy')
X_te_wed = np.load('preprocessed/X_te_wed_w20.npy')
X_te_fri = np.load('preprocessed/X_te_fri_w20.npy')
y_te_tue = np.load('preprocessed/y_te_tue_w20.npy', allow_pickle=True)
y_te_wed = np.load('preprocessed/y_te_wed_w20.npy', allow_pickle=True)
y_te_fri = np.load('preprocessed/y_te_fri_w20.npy', allow_pickle=True)

TIMESTEPS = X_train.shape[1]  # 20
FEATURES  = X_train.shape[2]  # 78

print(f"Train: {X_train.shape} / Val: {X_val.shape}")
print(f"TIMESTEPS={TIMESTEPS}, FEATURES={FEATURES}")

In [ ]:
# ── Section 3. 공통 함수 ──────────────────────────────────────
def get_threshold(model, X_val, percentile=95):
    X_pred = model.predict(X_val, verbose=0)
    mse = np.mean(np.square(X_val - X_pred), axis=(1, 2))
    return np.percentile(mse, percentile)

def get_mse(model, X):
    X_pred = model.predict(X, verbose=0)
    return np.mean(np.square(X - X_pred), axis=(1, 2))

def evaluate(model, threshold, name):
    results = {}
    for X_te, y_te, attack in [
        (X_te_tue, y_te_tue, 'BruteForce'),
        (X_te_wed, y_te_wed, 'DoS'),
        (X_te_fri, y_te_fri, 'DDoS'),
    ]:
        mse    = get_mse(model, X_te)
        y_pred = (mse > threshold).astype(int)
        y_true = (y_te != 'BENIGN').astype(int)

        report    = classification_report(y_true, y_pred, output_dict=True)
        recall    = report['1']['recall']    if '1' in report else 0
        precision = report['1']['precision'] if '1' in report else 0
        f1        = report['1']['f1-score']  if '1' in report else 0

        results[attack] = {'recall': recall, 'precision': precision, 'f1': f1}
        print(f"  [{name}] {attack} → Recall: {recall:.3f}  Precision: {precision:.3f}  F1: {f1:.3f}")
    return results

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
# ── Section 4. LSTM Autoencoder ───────────────────────────────
print("\n========== EXP1: LSTM ==========")

inp     = Input(shape=(TIMESTEPS, FEATURES))
encoded = LSTM(64, activation='relu', return_sequences=False)(inp)
repeated= RepeatVector(TIMESTEPS)(encoded)
decoded = LSTM(64, activation='relu', return_sequences=True)(repeated)
out     = TimeDistributed(Dense(FEATURES))(decoded)
lstm_model = Model(inp, out)
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.summary()

start = time.time()
lstm_hist = lstm_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
lstm_time = (time.time() - start) / len(lstm_hist.history['loss'])
lstm_threshold = get_threshold(lstm_model, X_val)
print(f"LSTM 임계값: {lstm_threshold:.6f} / 에폭당 평균: {lstm_time:.2f}s")
lstm_results = evaluate(lstm_model, lstm_threshold, 'LSTM')

In [ ]:
# ── Section 5. Bi-LSTM Autoencoder ───────────────────────────
print("\n========== EXP2: Bi-LSTM ==========")

inp     = Input(shape=(TIMESTEPS, FEATURES))
encoded = Bidirectional(LSTM(64, activation='relu', return_sequences=False))(inp)
repeated= RepeatVector(TIMESTEPS)(encoded)
decoded = Bidirectional(LSTM(64, activation='relu', return_sequences=True))(repeated)
out     = TimeDistributed(Dense(FEATURES))(decoded)
bilstm_model = Model(inp, out)
bilstm_model.compile(optimizer='adam', loss='mse')
bilstm_model.summary()

start = time.time()
bilstm_hist = bilstm_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
bilstm_time = (time.time() - start) / len(bilstm_hist.history['loss'])
bilstm_threshold = get_threshold(bilstm_model, X_val)
print(f"Bi-LSTM 임계값: {bilstm_threshold:.6f} / 에폭당 평균: {bilstm_time:.2f}s")
bilstm_results = evaluate(bilstm_model, bilstm_threshold, 'Bi-LSTM')

In [ ]:
# ── Section 6. Graph 1. Loss Curve ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, hist, name in zip(axes,
    [lstm_hist, bilstm_hist],
    ['LSTM', 'Bi-LSTM']):
    ax.plot(hist.history['loss'],     label='Train Loss')
    ax.plot(hist.history['val_loss'], label='Val Loss')
    ax.set_title(f'{name} Loss Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend()
plt.tight_layout()
plt.savefig('loss_curve_tier2.png', dpi=150)
plt.show()

In [ ]:
# ── Section 7. Graph 3. Scatter (Speed vs Recall) ─────────────
avg_recall = lambda r: np.mean([r[k]['recall'] for k in r])

names  = ['LSTM', 'Bi-LSTM']
times  = [lstm_time, bilstm_time]
recalls= [avg_recall(lstm_results), avg_recall(bilstm_results)]
colors = ['steelblue', 'tomato']

plt.figure(figsize=(7, 5))
for n, t, r, c in zip(names, times, recalls, colors):
    plt.scatter(t, r, s=200, color=c, label=n, zorder=5)
    plt.annotate(n, (t, r), textcoords='offset points',
                 xytext=(8, 4), fontsize=12)
plt.xlabel('에폭당 추론 시간 (s)')
plt.ylabel('평균 Recall')
plt.title('2차 방어선: Speed vs Recall')
plt.legend()
plt.tight_layout()
plt.savefig('scatter_tier2.png', dpi=150)
plt.show()

In [ ]:
# ── Section 8. 모델 저장 + 다운로드 ──────────────────────────
os.makedirs('models', exist_ok=True)
lstm_model.save('models/lstm_model.keras')
bilstm_model.save('models/bilstm_model.keras')

np.save('models/lstm_threshold.npy',   lstm_threshold)
np.save('models/bilstm_threshold.npy', bilstm_threshold)

shutil.make_archive('tier2_models', 'zip', 'models')
files.download('tier2_models.zip')
files.download('loss_curve_tier2.png')
files.download('scatter_tier2.png')
print("저장 및 다운로드 완료!")

# 시각화 (KDE, Waterfall)

1. Graph 2: KDE MSE 분포도
2. Graph 4: Waterfall 흐름도

In [ ]:
# ── Graph 2. KDE MSE 분포도 ───────────────────────────────────
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, threshold, name in zip(
    axes,
    [cnn_model,  bilstm_model],
    [cnn_threshold, bilstm_threshold],
    ['1D-CNN (1차)', 'Bi-LSTM (2차)']
):
    # 정상 MSE
    mse_normal = get_mse(model, X_te_tue[y_te_tue == 'BENIGN'])

    # 공격 MSE (3개 합치기)
    mse_attack = np.concatenate([
        get_mse(model, X_te_tue[y_te_tue != 'BENIGN']),
        get_mse(model, X_te_wed[y_te_wed != 'BENIGN']),
        get_mse(model, X_te_fri[y_te_fri != 'BENIGN']),
    ])

    # KDE
    for mse, label, color in [
        (mse_normal, 'Normal', 'steelblue'),
        (mse_attack, 'Attack', 'tomato')
    ]:
        kde = gaussian_kde(mse)
        x = np.linspace(mse.min(), mse.max(), 300)
        ax.plot(x, kde(x), label=label, color=color)
        ax.fill_between(x, kde(x), alpha=0.3, color=color)

    ax.axvline(threshold, color='black', linestyle='--',
               label=f'Threshold: {threshold:.4f}')
    ax.set_title(f'{name} MSE 분포')
    ax.set_xlabel('MSE')
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.savefig('kde_mse.png', dpi=150)
plt.show()

In [ ]:
# ── Graph 4. Waterfall 흐름도 ─────────────────────────────────
total     = 100000
tier1_pass   = int(total * 0.88)   # 1차에서 정상 처리
tier1_block  = int(total * 0.05)   # 1차에서 공격 차단
tier2_input  = total - tier1_pass - tier1_block
tier2_block  = int(tier2_input * 0.80)
tier2_pass   = tier2_input - tier2_block

labels = [
    '전체 트래픽',
    '1차 정상 통과\n(1D-CNN)',
    '1차 공격 차단\n(1D-CNN)',
    '2차 심층 검사\n(Bi-LSTM)',
    '2차 공격 차단\n(Bi-LSTM)',
    '최종 정상 통과'
]
values = [total, tier1_pass, tier1_block, tier2_input, tier2_block, tier2_pass]
colors = ['steelblue', 'seagreen', 'tomato', 'steelblue', 'tomato', 'seagreen']

plt.figure(figsize=(12, 5))
bars = plt.bar(labels, values, color=colors, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 500,
             f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.title('2-Tier 방어 시스템 Waterfall Throughput', fontsize=14)
plt.ylabel('트래픽 수')
plt.xticks(fontsize=9)
plt.tight_layout()
plt.savefig('waterfall.png', dpi=150)
plt.show()

print("시각화 완료!")